# Human approval and permissions

## Northstar Cloud: prepare a rollback, but never execute it autonomously

European checkout failures follow `deploy-1842`. The agent can collect evidence and draft a rollback, but an authorized incident commander must review the exact scope and make the decision. This notebook pairs theory and a deterministic, credential-free implementation.

**Learning outcomes:** design capabilities, produce a high-quality approval payload, persist and resume safely, handle approve/modify/reject/escalate paths, and make execution idempotent.


## Lifecycle

![Approval and permission lifecycle](assets/approval-permissions-lifecycle.svg)

The static SVG works in GitHub and local Jupyter without a Mermaid or JavaScript renderer. See the README for editable Mermaid diagrams.


## 1. Four different controls

Authentication answers *who are you?* Authorization answers *may that identity use this capability?* Approval answers *has an authorized human consented to this precise action now?* Execution answers *can the action occur once, safely?* A system prompt does none of these reliably.

The application owns these checks. The model may propose an action; it cannot grant itself permission.


In [ ]:
from lab import (
    ApprovalStore, ProposedAction, TOOL_PERMISSIONS, REQUIRED_REVIEWER,
    permission_matrix, propose_rollback, decide, get_evidence
)

for row in permission_matrix():
    print(row)


## 2. Least privilege starts with tool contracts

Northstar separates READ (`query_region_logs`), PROPOSE (`prepare_rollback`), and EXECUTE-WITH-APPROVAL (`rollback_deployment`). This makes both policy and review understandable. A broad `admin(command)` tool would make the permission decision ambiguous and difficult to audit.

An approval request includes tenant scope, exact typed arguments, risk, independent evidence IDs, expiry, and an action fingerprint. A vague `Approve?` dialog is an anti-pattern: it encourages rubber-stamping and cannot establish what was authorized.


In [ ]:
store = ApprovalStore()
request = propose_rollback(store)
print('paused:', request.status)
print('tool:', request.action.tool)
print('arguments:', request.action.arguments)
print('evidence:', [item.source for item in request.evidence])
print('fingerprint:', request.action_fingerprint)
print('required reviewer:', REQUIRED_REVIEWER[request.action.tool])


## 3. Approval is a persisted interruption

A reviewer may respond much later. Persist an opaque run ID, action, evidence, tenant, expiry, and status; load it only after tenant authorization. In LangGraph, `interrupt()` writes state through a checkpointer and `Command(resume=...)` continues with the same `thread_id`. The resumed node starts over, so code before an interrupt must be idempotent.

```python
from langgraph.types import Command, interrupt
def approval_node(state):
    response = interrupt({"action": state["action"], "evidence": state["evidence"]})
    return {"review_decision": response}
# graph.invoke(Command(resume={"decision": "approve"}), config=same_thread_config)
```

Do not loop repeatedly around `interrupt()` in one node. Save invalid input in state and route to a new invocation instead.


## 4. Experiment A — approval creates one executable record

The lab’s executor is simulated. It returns an execution record containing an idempotency key; it does not change production. Notice that the reason, actor, original fingerprint, final fingerprint, tenant, and timestamp land in the audit event.


In [ ]:
result = decide(
    store, request.run_id, 'northstar', 'incident_commander', 'approve',
    'Logs and deployment history support a scoped eu-west rollback.'
)
print('executed:', result['executed'])
print('execution:', result['execution'])
print('audit:', result['audit'])


## 5. Experiment B — rejection and invalid modification

Rejection is an intentional outcome, not an error that an agent should work around. It records the decision and routes the system to further investigation or escalation. An edited action must be validated as if it were new: an incident commander cannot silently broaden `eu-west` to `global`, nor change the validated deployment target.


In [ ]:
reject_store = ApprovalStore()
reject_request = propose_rollback(reject_store, run_id='reject-demo')
rejected = decide(
    reject_store, reject_request.run_id, 'northstar', 'incident_commander', 'reject',
    'Wait for the deployment owner to assess payment-service dependencies.'
)
print(rejected)

bad_store = ApprovalStore()
bad_request = propose_rollback(bad_store, run_id='bad-modification')
too_broad = ProposedAction(
    tool='rollback_deployment',
    arguments={'service': 'checkout', 'deployment_id': 'deploy-1842', 'region': 'global'},
    rationale='This deliberately tries to broaden the approved scope.', risk='high',
    evidence_ids=('service_health', 'logs', 'deployments'),
)
try:
    decide(bad_store, bad_request.run_id, 'northstar', 'incident_commander', 'modify', 'Test scope validation.', too_broad)
except ValueError as exc:
    print('blocked unsafe modification:', exc)


## 6. Failure modes and recovery

| Failure | Unsafe response | Safer design |
| --- | --- | --- |
| UI refresh resubmits approval | dispatch action again | durable idempotency fingerprint + status lookup |
| request is stale | execute with old evidence | expiry and revalidation of deployment state |
| wrong tenant loads request | expose action/evidence | tenant-scoped store lookup before rendering |
| model asks repeatedly | pressure reviewer | rate/budget limits and a safe escalation terminal state |
| reviewer edits scope | trust free-form text | typed schema, reauthorization, and risk reclassification |
| executor timeout | blind retry | idempotency key and reconciliation query |

Human review should be used for high-impact, ambiguous, or policy-sensitive decisions—not every low-risk read. Excessive approval prompts cause fatigue and encourage automatic approval.


## 7. Production readiness

Before connecting real tools: authenticate reviewers in your app; authorize by tenant, role, action, environment, and time; redact sensitive evidence; use durable checkpointers/stores; set expiry; make executor calls idempotent; log a tamper-resistant audit trail; track approval latency and override rates; and test cross-tenant access, prompt injection, stale approvals, policy bypasses, and retries.

### Exercises

1. Require a communications reviewer for `send_customer_notice`, even after rollback approval.
2. Add a two-person rule for a production payment rollback.
3. Add an expiry check and a revalidation tool for a new deployment.
4. Prove that `tenant-b` cannot load Northstar’s request.
5. Compare an automatic rollback workflow with this approval-gated route. Which evidence and risk threshold justify the added delay?


## References

- [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangChain human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)
- [OWASP GenAI Security Project](https://genai.owasp.org/)
